<a href="https://colab.research.google.com/github/rcNibedita/De-Novo-Gen/blob/main/01_Preprocess_ChEMBL_TAK1_Bioactivity_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount the Drive (Repeat everytime you create a notebook)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Relocate to the working directory
!ls "/content/drive/MyDrive/De-Novo"
%cd "/content/drive/MyDrive/De-Novo"
!pwd

01_Preprocess_ChEMBL_TAK1_Bioactivity_Data.ipynb  TAK1_clean_canonical.csv
chembl_raw.csv					  TAK1_clean.csv
/content/drive/MyDrive/De-Novo
/content/drive/MyDrive/De-Novo


In [ ]:
# # File debugging snippet; View the first few lines of the file incase simple pd.read_csv does not load it properly
# with open("chembl_raw.csv", "r", encoding="utf-8") as f:
#     for i in range(15):
#         print(f.readline())

"Molecule ChEMBL ID";"Molecule Name";"Molecule Max Phase";"Molecular Weight";"#RO5 Violations";"AlogP";"Compound Key";"Smiles";"Standard Type";"Standard Relation";"Standard Value";"Standard Units";"pChEMBL Value";"Data Validity Comment";"Comment";"Uo Units";"Ligand Efficiency BEI";"Ligand Efficiency LE";"Ligand Efficiency LLE";"Ligand Efficiency SEI";"Potential Duplicate";"Assay ChEMBL ID";"Assay Description";"Assay Type";"BAO Format ID";"BAO Label";"Assay Organism";"Assay Tissue ChEMBL ID";"Assay Tissue Name";"Assay Cell Type";"Assay Subcellular Fraction";"Assay Parameters";"Assay Variant Accession";"Assay Variant Mutation";"Target ChEMBL ID";"Target Name";"Target Organism";"Target Type";"Document ChEMBL ID";"Source ID";"Source Description";"Document Journal";"Document Year";"Cell ChEMBL ID";"Properties";"Action Type";"Standard Text Value";"Value"

"CHEMBL2407792";"";"None";"446.47";"0";"4.03";"12z";"CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc([N+](=O)[O-])c5)cc34)cn2)CC1";"IC50";"'='";"20

In [ ]:
# ============================================================
# Install Required Packages
# ============================================================
!pip -q install rdkit

In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================

import pandas as pd
import numpy as np

from rdkit import Chem

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ============================================================
# Load the ChEMBL Bioactivity Dataset
# ============================================================

# If using Google Drive
# df = pd.read_csv("/content/drive/MyDrive/De-Novo/chembl_raw.csv")

# If uploaded directly into Colab
df = pd.read_csv("chembl_raw.csv",
                 sep=";",
                 quotechar='"')

print("Dataset loaded successfully.")
print("Number of records:", len(df))

Dataset loaded successfully.
Number of records: 343


In [ ]:
# ============================================================
# Inspect Dataset
# ============================================================

print(df.head())

print("\nColumns available:\n")
print(df.columns.tolist())

  Molecule ChEMBL ID Molecule Name  Molecule Max Phase  Molecular Weight  \
0      CHEMBL2407792           NaN                 NaN            446.47   
1      CHEMBL2407759           NaN                 NaN            459.54   
2      CHEMBL4555884           NaN                 NaN            484.09   
3      CHEMBL4446434           NaN                 NaN            358.20   
4      CHEMBL2407763           NaN                 NaN            457.56   

   #RO5 Violations  AlogP Compound Key  \
0                0   4.03          12z   
1                0   4.13         12az   
2                1   5.19            1   
3                0   4.59            4   
4                1   5.34         12bd   

                                              Smiles Standard Type  \
0  CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc([N+](=O...          IC50   
1  CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc6nnsc56)...          IC50   
2     COc1cc(Br)cc2[nH]cc(-c3oc(-c4ccc[nH]4)nc3I)c12          IC50   
3      COc

In [ ]:
# ============================================================
# Inspect Bioactivity Columns
# ============================================================

bioactivity = df[
    [
        "Molecule ChEMBL ID",
        "Smiles",
        "Standard Type",
        "Standard Relation",
        "Standard Value",
        "Standard Units",
        "pChEMBL Value"
    ]
]

bioactivity.head(20)

,Molecule ChEMBL ID,Smiles,Standard Type,Standard Relation,Standard Value,Standard Units,pChEMBL Value
0,CHEMBL2407792,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc([N+](=O...,IC50,'=',200.0,nM,6.70
1,CHEMBL2407759,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc6nnsc56)...,IC50,'=',12.0,nM,7.92
2,CHEMBL4555884,COc1cc(Br)cc2[nH]cc(-c3oc(-c4ccc[nH]4)nc3I)c12,IC50,'>',10000.0,nM,NaN
3,CHEMBL4446434,COc1cc(Br)cc2[nH]cc(-c3cnc(-c4ccc[nH]4)o3)c12,IC50,'=',1000.0,nM,6.00
4,CHEMBL2407763,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5csc6ccccc56)...,IC50,'=',440.0,nM,6.36
5,CHEMBL3426225,CCn1nc(C#Cc2cc(C(=O)Nc3ccc(CN4CCN(C)CC4)c(C(F)...,IC50,'=',61.0,nM,7.21
6,CHEMBL4787515,Cc1ccc(NC(=O)c2cccc(C(F)(F)F)c2)cc1C#Cc1nn(C2C...,IC50,'=',203.0,nM,6.69
7,CHEMBL2407770,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc(C(N)=O)...,IC50,'=',3100.0,nM,5.51
8,CHEMBL2407772,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc(CO)c5)c...,IC50,'=',470.0,nM,6.33
9,CHEMBL2407767,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc(C#N)c5)...,IC50,'=',370.0,nM,6.43


In [ ]:
# ============================================================
# Inspect Missing Bioactivity Values
# ============================================================
print(df[df["Standard Units"].isna()][[
    "Molecule ChEMBL ID",
    "Standard Type",
    "Standard Relation",
    "Standard Value",
    "Standard Units"
]])

print()

print("Missing Standard Units")
print(df["Standard Units"].isna().sum())

print("Missing SMILES")
print(df["Smiles"].isna().sum())

    Molecule ChEMBL ID Standard Type Standard Relation  Standard Value  \
18       CHEMBL2407790          IC50               NaN             NaN   
36       CHEMBL4460150          IC50               NaN             NaN   
44       CHEMBL2407754          IC50               NaN             NaN   
45       CHEMBL2407776          IC50               NaN             NaN   
59       CHEMBL2407782          IC50               NaN             NaN   
60       CHEMBL2407780          IC50               NaN             NaN   
154      CHEMBL2407764          IC50               NaN             NaN   
177      CHEMBL4569421          IC50               NaN             NaN   
178      CHEMBL4462060          IC50               NaN             NaN   

    Standard Units  
18             NaN  
36             NaN  
44             NaN  
45             NaN  
59             NaN  
60             NaN  
154            NaN  
177            NaN  
178            NaN  

Missing Standard Units
9
Missing SMILES
0


In [ ]:
# ============================================================
# Inspect Bioactivity Type, Units & Relations
# ============================================================

# Confirm if all instances are IC50 type
print(df["Standard Type"].value_counts())

# Count and display different instances of Standard Relation
print(df["Standard Relation"].value_counts(dropna=False))

# Display different instances of Standard Units
print("Standard Units:", df["Standard Units"].unique())

Standard Type
IC50    343
Name: count, dtype: int64
Standard Relation
'='    299
'>'     33
NaN      9
'<'      1
'~'      1
Name: count, dtype: int64
Standard Units: ['nM' nan "10'-4nM"]


In [ ]:
# ===================================================================
# Inspect bioactivity-related columns with specific erroneous values
# ===================================================================
# df[df["Standard Units"] == "10'-4nM"][
#     [
#         "Molecule ChEMBL ID",
#         "Standard Type",
#         "Standard Relation",
#         "Standard Value",
#         "Standard Units",
#         "Assay Description",
#         "Document ChEMBL ID"
#     ]
# ]

In [ ]:
# ============================================================
# Keep Potent Compounds (IC50 ≤ 1000 nM)
# ============================================================
df_active = df[
    (df["Standard Type"] == "IC50") &
    (df["Standard Units"] == "nM") &
    (df["Standard Value"] <= 1000) &
    (df["Standard Relation"].isin(["'='", "'<'"]))
].copy()
print(f"Number of compounds retained: {len(df_active)}")
df_active.head()

Number of compounds retained: 226


,Molecule ChEMBL ID,Molecule Name,Molecule Max Phase,Molecular Weight,#RO5 Violations,AlogP,Compound Key,Smiles,Standard Type,Standard Relation,...,Document ChEMBL ID,Source ID,Source Description,Document Journal,Document Year,Cell ChEMBL ID,Properties,Action Type,Standard Text Value,Value
0,CHEMBL2407792,NaN,NaN,446.47,0,4.03,12z,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc([N+](=O...,IC50,'=',...,CHEMBL2407040,1,Scientific Literature,Bioorg Med Chem Lett,2013,CHEMBL3308372,NaN,NaN,NaN,0.200
1,CHEMBL2407759,NaN,NaN,459.54,0,4.13,12az,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5cccc6nnsc56)...,IC50,'=',...,CHEMBL2407040,1,Scientific Literature,Bioorg Med Chem Lett,2013,CHEMBL3308372,NaN,NaN,NaN,0.012
3,CHEMBL4446434,NaN,NaN,358.20,0,4.59,4,COc1cc(Br)cc2[nH]cc(-c3cnc(-c4ccc[nH]4)o3)c12,IC50,'=',...,CHEMBL4334516,1,Scientific Literature,J Med Chem,2019,NaN,NaN,NaN,NaN,1.000
4,CHEMBL2407763,NaN,NaN,457.56,1,5.34,12bd,CC(=O)N1CCC(n2cc(-c3cnc(N)c4oc(-c5csc6ccccc56)...,IC50,'=',...,CHEMBL2407040,1,Scientific Literature,Bioorg Med Chem Lett,2013,CHEMBL3308372,NaN,NaN,NaN,0.440
5,CHEMBL3426225,NaN,NaN,576.63,1,4.16,BDBM50086441,CCn1nc(C#Cc2cc(C(=O)Nc3ccc(CN4CCN(C)CC4)c(C(F)...,IC50,'=',...,CHEMBL5726853,37,BindingDB Patent Bioactivity Data,NaN,2019,NaN,NaN,NaN,NaN,61.000


In [ ]:
# ============================================================
# Remove Molecules Without SMILES
# ============================================================
before = len(df_active)

df_active = df_active.dropna(
    subset=["Smiles"]
)

print("Removed null values:", before-len(df_active))
print("Remaining:", len(df_active))

# ============================================================
# Remove Duplicate SMILES
# ============================================================

before = len(df_active)

df_active = df_active.drop_duplicates(
    subset=["Smiles"]
)

print("Duplicates removed:", before-len(df_active))
print("Remaining:", len(df_active))

# ============================================================
# Export the Cleaned File
# ============================================================
df_active.to_csv("TAK1_clean.csv",index=False)

Removed null values: 0
Remaining: 226
Duplicates removed: 48
Remaining: 178


In [ ]:
# ============================================================
# Canonicalize SMILES
# ============================================================
def canonicalize(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

df_active["Canonical_SMILES"] = df_active["Smiles"].apply(canonicalize)
print("Canonicalization completed.")

Canonicalization completed.


In [ ]:
# ============================================================
# Remove Invalid Molecules
# ============================================================
before = len(df_active)

df_active = df_active.dropna(
    subset=["Canonical_SMILES"]
)

print("Invalid molecules removed:", before-len(df_active))
print("Remaining:", len(df_active))

# ============================================================
# Remove Duplicate Canonical Molecules
# ============================================================

before = len(df_active)

df_active = df_active.drop_duplicates(
    subset=["Canonical_SMILES"]
)

print("Canonical duplicates removed:",
      before-len(df_active))

print("Final dataset size:",
      len(df_active))

Invalid molecules removed: 0
Remaining: 178
Canonical duplicates removed: 0
Final dataset size: 178


In [ ]:
# ============================================================
# Inspect Final Dataset
# ============================================================

df_active.head()

# ============================================================
# Export Clean Dataset
# ============================================================

df_active.to_csv(
    "TAK1_clean_canonical.csv",
    index=False
)

print("TAK1_clean_canonical.csv saved successfully.")

TAK1_clean_canonical.csv saved successfully.


In [ ]:
# ============================================================
# Dataset Summary
# ============================================================

print("="*50)

print("FINAL DATASET SUMMARY")

print("="*50)

print("Total compounds:",
      len(df_active))

print()

print("Columns:")

print(df_active.columns.tolist())

FINAL DATASET SUMMARY
Total compounds: 178

Columns:
['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate', 'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID', 'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation', 'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type', 'Document ChEMBL ID', 'Source ID', 'Source Description', 'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties', 'Action Type', 'Standard Text Value', 'Value', 'Canonical_SMILES']
